# cAPTure: XGB-P development training

This CPU-only notebook trains the packet-only XGBoost baseline under the two frozen benign-background folds. It uses completed canonical packet and fold-local preprocessing artifacts on Drive. Each development packet receives exactly one out-of-fold (OOF) score. No test scenario is read, no threshold is selected, and no final five-scenario model is trained here.

Run sections 1–6 for the primary depth-5 configuration. Section 7 is the optional, more expensive depth-10 capacity sensitivity. A completed fold is verified and reused if the same run ID is resumed; an incomplete output is never overwritten automatically.

## 1. Mount Drive and load the current repository

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_WORK_ROOT = Path("/content/capture_xgb_p_work")

if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")
required_files = ["code/python/utils/capture_xgb_p.py", "code/python/tests/test_capture_xgb_p.py", "code/python/requirements-capture-xgb.txt", "configs/capture_experiment_v1.yaml", "configs/capture_packet_schema_v1.yaml", "configs/capture_preprocessing_v1.yaml"]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "code/python/requirements-capture-xgb.txt")], check=True)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))
print("CPU XGB-P environment is ready.")
print("Repository commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip())

## 2. Run synthetic protocol checks

In [ ]:
test_environment = dict(os.environ)
test_environment["PYTHONPATH"] = str(PROJECT_ROOT / "code/python")
for pattern in ("test_capture_preprocess.py", "test_capture_xgb_p.py"):
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(PROJECT_ROOT / "code/python/tests"), "-p", pattern, "-v"], env=test_environment, cwd=PROJECT_ROOT, check=True)
print("Synthetic preprocessing and XGB-P checks passed.")

## 3. Bind the approved development artifacts and run ID

Set `RUN_ID` to an earlier XGB-P run ID when resuming after a Colab disconnect. A new ID starts a new run; old results are never overwritten.

In [ ]:
from IPython.display import display
import pandas as pd
from utils.capture_data import load_manifest
from utils.capture_xgb_p import run_xgb_p_fold, summarize_xgb_p_oof, validate_xgb_p_fold_run

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
PACKET_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml"
PREPROCESSING_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_preprocessing_v1.yaml"
PREPARED_RUN_DIR = DRIVE_ROOT / "prepared_runs" / "20260917T235058_827743Z_prepare_full_dev"
PREPROCESSING_AUDIT_DIR = DRIVE_ROOT / "preprocessing_runs" / "20260919T004143_161396Z_preprocessing"
RUN_ID = None  # Replace with a previous XGB-P run ID only when resuming.
RUN_DEPTH10_SENSITIVITY = False  # Decide from CPU budget before inspecting model metrics.
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_xgb_p"
DRIVE_RUN_DIR = DRIVE_ROOT / "xgb_p_runs" / RUN_ID
BATCH_SIZE = 50_000
CPU_THREADS = 2

for path in (PREPARED_RUN_DIR, PREPROCESSING_AUDIT_DIR):
    if not path.is_dir():
        raise FileNotFoundError(f"Required Drive run is missing: {path}")
manifest = load_manifest(MANIFEST_PATH)
assert manifest["training"]["hyperparameter_search"]["xgb_p"]["primary_configuration"] == "depth5_primary"
print("Prepared input:", PREPARED_RUN_DIR)
print("Preprocessing audit:", PREPROCESSING_AUDIT_DIR)
print("XGB-P run ID:", RUN_ID)
print("XGB-P Drive output:", DRIVE_RUN_DIR)
print("Depth-10 sensitivity planned:", RUN_DEPTH10_SENSITIVITY)

## 4. Check local storage before training

The large fold-B feature matrix is temporary and stays on Colab local storage. Model files, fold reports, and OOF packet scores are copied to Drive and verified. The runner rechecks source hashes, preprocessing provenance, row counts, and free local storage.

In [ ]:
LOCAL_WORK_ROOT.mkdir(parents=True, exist_ok=True)
local_free_gib = shutil.disk_usage(LOCAL_WORK_ROOT).free / (1024 ** 3)
drive_free_gib = shutil.disk_usage(DRIVE_ROOT).free / (1024 ** 3)
training_rows = {fold: sum(json.loads((PREPARED_RUN_DIR / scenario / "preparation_report.json").read_text(encoding="utf-8"))["counts"]["packets"] for scenario in split["train"]) for fold, split in manifest["validation"]["folds"].items()}
display(pd.DataFrame([{"fold": fold, "training_packets": rows, "estimated_feature_matrix_gib": rows * 103 * 4 / (1024 ** 3)} for fold, rows in training_rows.items()]))
print(f"Free local storage: {local_free_gib:.1f} GiB")
print(f"Free Drive storage: {drive_free_gib:.1f} GiB")
if local_free_gib < max(rows * 103 * 4 / (1024 ** 3) for rows in training_rows.values()) + 2:
    raise OSError("Local storage is too small for the fold-B feature matrix.")

## 5. Train the primary depth-5 configuration

Each fold can take a long time on Colab CPU, especially fold B. Run fold A first, inspect its report, then run fold B. An existing complete fold is checksum-verified instead of retrained.

In [ ]:
def train_or_verify(fold, configuration_name):
    output_dir = DRIVE_RUN_DIR / configuration_name / f"fold_{fold}"
    if output_dir.exists():
        report = validate_xgb_p_fold_run(output_dir, fold, configuration_name)
        print(f"Verified existing {configuration_name} fold {fold}: {output_dir}")
        return report
    return run_xgb_p_fold(manifest_path=MANIFEST_PATH, packet_schema_path=PACKET_SCHEMA_PATH, preprocessing_schema_path=PREPROCESSING_SCHEMA_PATH, prepared_run_dir=PREPARED_RUN_DIR, preprocessing_audit_dir=PREPROCESSING_AUDIT_DIR, output_dir=output_dir, local_work_root=LOCAL_WORK_ROOT, fold=fold, configuration_name=configuration_name, batch_size=BATCH_SIZE, nthread=CPU_THREADS)

primary_a = train_or_verify("A", "depth5_primary")
display(pd.DataFrame.from_dict(primary_a["validation"], orient="index")[["rows", "normal_packets", "attack_packets", "packet_roc_auc", "packet_pr_auc_diagnostic"]])

In [ ]:
primary_b = train_or_verify("B", "depth5_primary")
display(pd.DataFrame.from_dict(primary_b["validation"], orient="index")[["rows", "normal_packets", "attack_packets", "packet_roc_auc", "packet_pr_auc_diagnostic"]])

## 6. Review the complete primary OOF result

In [ ]:
primary_summary = summarize_xgb_p_oof(DRIVE_RUN_DIR, "depth5_primary")
display(pd.DataFrame.from_dict(primary_summary["scenario_metrics"], orient="index"))
print("Fold ROC-AUC:", primary_summary["fold_packet_roc_auc"])
print("Hierarchical macro OOF packet ROC-AUC:", primary_summary["hierarchical_macro_oof_packet_roc_auc"])
print("Thresholds selected:", primary_summary["thresholds_selected"])

## 7. Optional depth-10 capacity sensitivity

Decide whether the CPU and memory budget permits this sensitivity before inspecting the primary validation metrics. The section can be executed later for practical reasons, but its activation must not depend on model performance. Depth 10 is one of the values in the authors' XGBoost grid. It is not an automatic replacement for the predeclared depth-5 primary comparison. Keep both fold runs under the same run ID.

In [ ]:
if RUN_DEPTH10_SENSITIVITY:
    sensitivity_a = train_or_verify("A", "depth10_sensitivity")
    sensitivity_b = train_or_verify("B", "depth10_sensitivity")
    sensitivity_summary = summarize_xgb_p_oof(DRIVE_RUN_DIR, "depth10_sensitivity")
    display(pd.DataFrame([{"configuration": "depth5_primary", "macro_oof_packet_roc_auc": primary_summary["hierarchical_macro_oof_packet_roc_auc"]}, {"configuration": "depth10_sensitivity", "macro_oof_packet_roc_auc": sensitivity_summary["hierarchical_macro_oof_packet_roc_auc"]}]))
else:
    print("Depth-10 sensitivity is disabled; the primary result remains complete.")